[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cioos-siooc/CPDW-VI/blob/main/docs/notebooks/02_obis_parquet_duckdb.ipynb)

> The version on the docs site is a static render — to actually run the cells, click the badge above (Google Colab) or follow [Setup](../workshop/setup.md) to run locally.

In [ ]:
# @colab-setup
# Run this cell first. On Colab it installs the deps; locally it is a no-op.
import sys
if "google.colab" in sys.modules:
    %pip install -q duckdb pandas requests


# 02 — OBIS Parquet on S3 via DuckDB

**Time budget:** ~15 min · **Goal:** the headline performance demo. Pull whole datasets in seconds, run analytical SQL directly against S3.

OBIS publishes per-dataset Parquet files (see [iobis/obis-open-data](https://github.com/iobis/obis-open-data)) at:

```
https://obis-open-data.s3.amazonaws.com/occurrence/{dataset_uuid}.parquet
```

No auth, no SDK. DuckDB scans the file over HTTPS using HTTP range requests — it only fetches the columns and row groups it needs. The schema has nested `interpreted` and `verbatim` structs; **always unwrap `interpreted.*`** unless you specifically want the raw, unparsed values.

This is what `explore-cioos/harvester/cde_harvester/obis_harvester.py:294–322` does in production.

In [1]:
import duckdb
import pandas as pd
import time

DEMO_UUID = 'd895e645-a98d-4720-b6fb-332929190f36'  # Maritimes Spring RV Surveys
PARQUET_BASE = 'https://obis-open-data.s3.amazonaws.com/occurrence'

con = duckdb.connect()
con.sql('INSTALL httpfs; LOAD httpfs;')
con.sql("SET s3_region='us-east-1';")
print('DuckDB version:', duckdb.__version__)

DuckDB version: 1.5.2


## 1. Schema discovery

Before querying, look at what's there. `DESCRIBE` plus `LIMIT 0` is cheap — it only fetches the Parquet footer.

In [2]:
url = f"{PARQUET_BASE}/{DEMO_UUID}.parquet"
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{url}') LIMIT 0").df()
schema

,column_name,column_type,null,key,default,extra
0,_id,VARCHAR,YES,None,None,None
1,_event_id,VARCHAR,YES,None,None,None
2,_occurrence_id,VARCHAR,YES,None,None,None
3,dataset_id,VARCHAR,YES,None,None,None
4,node_ids,VARCHAR[],YES,None,None,None
5,source,"STRUCT(acceptedNameUsage VARCHAR, acceptedName...",YES,None,None,None
6,interpreted,"STRUCT(acceptedNameUsage VARCHAR, acceptedName...",YES,None,None,None
7,extensions,"STRUCT(""http://rs.gbif.org/terms/1.0/DNADerive...",YES,None,None,None
8,missing,VARCHAR[],YES,None,None,None
9,invalid,VARCHAR[],YES,None,None,None


## 2. Single-dataset pull

Verbatim adaptation of the production query in `obis_harvester.py:294–319`. Note the `interpreted.*` unwrap and the lat/lon bounds filter (records outside the Web Mercator range break tile rendering).

In [3]:
occurrences_query = f"""
    SELECT
        interpreted.decimalLatitude       AS decimalLatitude,
        interpreted.decimalLongitude      AS decimalLongitude,
        interpreted.date_start            AS date_start,
        interpreted.date_end              AS date_end,
        interpreted.minimumDepthInMeters  AS minDepth,
        interpreted.maximumDepthInMeters  AS maxDepth,
        interpreted.scientificName        AS scientificName,
        _id                               AS id
    FROM read_parquet('{url}')
    WHERE interpreted.decimalLatitude  BETWEEN -85.06 AND 85.06
      AND interpreted.decimalLongitude BETWEEN -180   AND 180
"""

t0 = time.perf_counter()
df = con.sql(occurrences_query).df()
parquet_secs = time.perf_counter() - t0
print(f'Parquet pull: {len(df):,} records in {parquet_secs:.2f} s ({len(df)/parquet_secs:,.0f} rec/s)')
df.head()

Parquet pull: 650,166 records in 2.15 s (301,712 rec/s)


,decimalLatitude,decimalLongitude,date_start,date_end,minDepth,maxDepth,scientificName,id
0,40.231167,-68.820667,636163200000,636163200000,NaN,NaN,Pisces,73952c55-6a04-43b0-b69c-f4fa4b575d83
1,40.450667,-69.419667,1109462400000,1109462400000,NaN,NaN,Bothus,bf6612d6-855c-4bb1-a36d-ede18a526f2d
2,40.450667,-69.419667,1109462400000,1109462400000,NaN,NaN,Bothus,c1462459-7e24-4ffd-91f0-85b2aa3552bc
3,44.850000,-61.333333,321062400000,321062400000,NaN,NaN,Gadus morhua,be11fe1c-52ea-49dd-8913-981482ed0720
4,43.883333,-62.250000,322099200000,322099200000,NaN,NaN,Gadus morhua,abf67d47-b1ba-445f-8f5d-29723ac0c71a


## 3. Bench: Parquet vs paged REST

Same dataset, two routes, side-by-side timing. We cap the REST pull at 2 pages so the cell stays workshop-sized; the speed ratio holds in both directions.

In [4]:
import requests

API = 'https://api.obis.org/v3'

def fetch_via_rest(dataset_id: str, page_size: int = 10000, max_pages: int = 2):
    params = {'datasetid': dataset_id, 'size': page_size}
    out = []
    for _ in range(max_pages):
        r = requests.get(f'{API}/occurrence', params=params, timeout=120)
        r.raise_for_status()
        results = r.json().get('results', [])
        if not results:
            break
        out.extend(results)
        if len(results) < page_size:
            break
        params['after'] = results[-1].get('id')
    return out

t0 = time.perf_counter()
rest_records = fetch_via_rest(DEMO_UUID, max_pages=2)
rest_secs = time.perf_counter() - t0
print(f'REST  (2 pages): {len(rest_records):,} records in {rest_secs:.2f} s ({len(rest_records)/rest_secs:,.0f} rec/s)')
print(f'Parquet (full):  {len(df):,} records in {parquet_secs:.2f} s ({len(df)/parquet_secs:,.0f} rec/s)')
print(f'\nParquet is ~{(len(df)/parquet_secs)/(len(rest_records)/rest_secs):.1f}× faster on rec/s.')

REST  (2 pages): 20,000 records in 13.63 s (1,468 rec/s)
Parquet (full):  650,166 records in 2.15 s (301,712 rec/s)

Parquet is ~205.6× faster on rec/s.


## 4. Spatial query — Canadian Pacific bbox

Push the spatial filter into DuckDB instead of pulling everything and filtering in pandas. Cheap, even when the dataset is huge.

In [5]:
spatial_query = f"""
    SELECT interpreted.scientificName    AS scientificName,
           interpreted.decimalLatitude   AS lat,
           interpreted.decimalLongitude  AS lon
    FROM read_parquet('{url}')
    WHERE interpreted.decimalLatitude  BETWEEN 47 AND 60
      AND interpreted.decimalLongitude BETWEEN -140 AND -120
"""
pacific = con.sql(spatial_query).df()
print(f'{len(pacific):,} records in Canadian Pacific bbox')
pacific.head()

0 records in Canadian Pacific bbox


,scientificName,lat,lon


## 5. Aggregation — top species

SQL group-by on the remote Parquet — none of the rows ever land in pandas.

In [6]:
top_species = con.sql(f"""
    SELECT interpreted.scientificName AS scientificName,
           COUNT(*) AS n_records
    FROM read_parquet('{url}')
    WHERE interpreted.scientificName IS NOT NULL
    GROUP BY 1
    ORDER BY n_records DESC
    LIMIT 20
""").df()
top_species

,scientificName,n_records
0,Melanogrammus aeglefinus,139499
1,Merluccius bilinearis,61573
2,Gadus morhua,37493
3,Leucoraja ocellata,35597
4,Myoxocephalus octodecemspinosus,30336
5,Myzopsetta ferruginea,30048
6,Clupea harengus,29435
7,Leucoraja erinaceus,25957
8,Hippoglossoides platessoides,19921
9,Placopecten magellanicus,18739


## 6. Multi-dataset union

DuckDB's `read_parquet([...])` accepts a list of URLs and unions them into a single virtual table. This is how you'd start building a regional summary across many datasets.

In [7]:
# Pick a few small Canadian datasets for the demo (replace with your own UUIDs).
uuids = [
    DEMO_UUID,
    '4b5e4ccb-cf66-44e4-8890-fa68f8404c3f',
]
urls = [f'{PARQUET_BASE}/{u}.parquet' for u in uuids]
url_list_sql = ', '.join(f"'{u}'" for u in urls)

summary = con.sql(f"""
    SELECT _dataset_id AS dataset_id,
           COUNT(*) AS n_records,
           COUNT(DISTINCT interpreted.scientificName) AS n_species
    FROM read_parquet([{url_list_sql}], filename = '_dataset_id')
    GROUP BY 1
""").df()
summary

,dataset_id,n_records,n_species
0,https://obis-open-data.s3.amazonaws.com/occurr...,345,3
1,https://obis-open-data.s3.amazonaws.com/occurr...,650166,440


## 7. Mini grid aggregation

Lighter version of the 1/12° grid aggregation in `obis_harvester.py:137–203` — snap to a coarse grid and emit one row per cell.

In [8]:
GRID_DEG = 1 / 12  # ~5 nautical miles
grid = con.sql(f"""
    SELECT
        ROUND(interpreted.decimalLatitude  / {GRID_DEG}) * {GRID_DEG} AS lat_grid,
        ROUND(interpreted.decimalLongitude / {GRID_DEG}) * {GRID_DEG} AS lon_grid,
        COUNT(*) AS n_records,
        COUNT(DISTINCT interpreted.scientificName) AS n_species,
        MIN(interpreted.date_start) AS first_seen,
        MAX(interpreted.date_end)   AS last_seen
    FROM read_parquet('{url}')
    WHERE interpreted.decimalLatitude  BETWEEN -85.06 AND 85.06
      AND interpreted.decimalLongitude BETWEEN -180   AND 180
    GROUP BY 1, 2
""").df()
print(f'{len(grid):,} grid cells')
grid.head()

1,614 grid cells


,lat_grid,lon_grid,n_records,n_species,first_seen,last_seen
0,41.583333,-66.250000,2083,47,793065600000,1551830400000
1,42.000000,-66.750000,2739,57,574214400000,1456617600000
2,42.000000,-66.166667,2700,59,322790400000,1456272000000
3,41.583333,-66.583333,2604,48,574041600000,1551830400000
4,43.083333,-62.250000,102,9,322185600000,322185600000


## Gotchas to flag in the talk

- **Lag**: the Parquet exports trail the live API by hours/days — don't use Parquet for time-critical pulls.
- **Coverage**: a few private/embargoed datasets are API-only; if `read_parquet` 404s, fall back to REST (Notebook 01).
- **Schema**: always unwrap `interpreted.*`; the `verbatim.*` struct is the unparsed Darwin Core (useful for QC, not analysis).
- **Coordinate validity**: ±85.06° lat / ±180° lon bounds are essential before sending data to a Web Mercator map renderer.

**Next:** Notebook 03 — resolve those `scientificName` strings against WoRMS to get authoritative AphiaIDs and full taxonomic classifications.